# Base Language Model

The `base.py` module defines the shared abstract interface and common configuration used by LangChain language-model implementations.

It provides language-model input and output aliases, LangSmith tracing parameters, a cached fallback GPT-2 tokenizer, model-level cache and callback configuration, package-version tracing metadata, structured-output hooks, synchronous and asynchronous prompt-generation contracts, and token-counting utilities.

# LangSmithParams

`LangSmithParams` defines optional standardized metadata attached to language-model traces.

## Bases

- `TypedDict`
- `total=False`

## Attributes

1. `ls_provider`: Stores the model provider name.
   * **Type:**
     ```python
     ls_provider: str
     ```

2. `ls_model_name`: Stores the model name.
   * **Type:**
     ```python
     ls_model_name: str
     ```

3. `ls_model_type`: Stores whether the traced model is a chat model or a text-completion model.
   * **Type:**
     ```python
     ls_model_type: Literal[
         "chat",
         "llm"
     ]
     ```

4. `ls_temperature`: Stores the generation temperature when available.
   * **Type:**
     ```python
     ls_temperature: float | None
     ```

5. `ls_max_tokens`: Stores the maximum number of generated tokens when available.
   * **Type:**
     ```python
     ls_max_tokens: int | None
     ```

6. `ls_stop`: Stores the stop strings used for generation.
   * **Type:**
     ```python
     ls_stop: list[str] | None
     ```

7. `ls_integration`: Stores the integration responsible for creating the trace.
   * **Type:**
     ```python
     ls_integration: str
     ```

## Type Aliases and Type Variables

1. `LanguageModelInput`: Represents the input accepted by a language model.

   Input may be a `PromptValue`, a plain string, or a sequence of message-like values.

   * **Definition:**
     ```python
     LanguageModelInput = (
         PromptValue
         | str
         | Sequence[
             MessageLikeRepresentation
         ]
     )
     ```

2. `LanguageModelOutput`: Represents the standard output of a language model.
   * **Definition:**
     ```python
     LanguageModelOutput = BaseMessage | str
     ```

3. `LanguageModelLike`: Represents a Runnable-compatible language-model interface.
   * **Definition:**
     ```python
     LanguageModelLike = Runnable[
         LanguageModelInput,
         LanguageModelOutput
     ]
     ```

4. `LanguageModelOutputVar`: Represents the output type of a concrete language model.

   The type variable is constrained to `AIMessage` or `str`.

   * **Definition:**
     ```python
     LanguageModelOutputVar = TypeVar(
         "LanguageModelOutputVar",
         AIMessage,
         str
     )
     ```

### Functions

1. `get_tokenizer`: Returns a cached GPT-2 tokenizer instance.

   The tokenizer is loaded through `GPT2TokenizerFast.from_pretrained("gpt2")` only on the first successful call. Later calls reuse the cached instance.

   An `ImportError` is raised when the optional `transformers` package is unavailable.

   * **Syntax:**
     ```python
     get_tokenizer() -> Any
     ```

# BaseLanguageModel

`BaseLanguageModel` is the abstract base class for LangChain language-model wrappers.

It combines the Runnable serialization interface with shared cache, tracing, callback, metadata, structured-output, prompt-generation, and token-counting behaviour.

Concrete text-completion and chat-model classes must implement the synchronous and asynchronous prompt-generation methods.

## Bases

- `RunnableSerializable[LanguageModelInput, LanguageModelOutputVar]`
- `ABC`

## Attributes

1. `cache`: Controls response caching.

   - `True` uses the global cache.
   - `False` disables caching.
   - `None` uses the global cache only when one is configured.
   - A `BaseCache` instance uses that cache directly.

   Streaming model methods do not currently support caching.

   * **Type:**
     ```python
     cache: BaseCache | bool | None = Field(
         default=None,
         exclude=True
     )
     ```

2. `verbose`: Controls whether generated response text is printed.

   Its default value is obtained from LangChain's global verbosity setting. Supplying `None` also resolves to the global setting.

   * **Type:**
     ```python
     verbose: bool = Field(
         default_factory=_get_verbosity,
         exclude=True,
         repr=False
     )
     ```

3. `callbacks`: Stores callbacks added to model run traces.
   * **Type:**
     ```python
     callbacks: Callbacks = Field(
         default=None,
         exclude=True
     )
     ```

4. `tags`: Stores tags added to model run traces.
   * **Type:**
     ```python
     tags: list[str] | None = Field(
         default=None,
         exclude=True
     )
     ```

5. `metadata`: Stores metadata added to model run traces.

   The `lc_versions` entry is automatically populated after model initialization.

   * **Type:**
     ```python
     metadata: dict[
         str,
         Any
     ] | None = Field(
         default=None,
         exclude=True
     )
     ```

6. `custom_get_token_ids`: Stores an optional model-specific text-to-token-ID encoder.

   When provided, `get_token_ids` uses this function instead of the fallback GPT-2 tokenizer.

   * **Type:**
     ```python
     custom_get_token_ids: Callable[
         [str],
         list[int]
     ] | None = Field(
         default=None,
         exclude=True
     )
     ```

## Configuration

1. `model_config`: Allows arbitrary Python types in the Pydantic model.
   * **Definition:**
     ```python
     model_config = ConfigDict(
         arbitrary_types_allowed=True
     )
     ```

### Properties

1. `InputType`: Returns the concrete Runnable input union used to generate an informative input schema.

   The returned union replaces the abstract `BaseMessage` type with supported concrete message representations.

   * **Type:**
     ```python
     InputType: TypeAlias
     ```

   * **Value:**
     ```python
     str
     | StringPromptValue
     | ChatPromptValueConcrete
     | list[AnyMessage]
     ```

2. `_identifying_params`: Returns the model attributes used to identify the serialized model.

   The base implementation returns `lc_attributes`.

   * **Type:**
     ```python
     _identifying_params: Mapping[
         str,
         Any
     ]
     ```

### Methods

1. `model_post_init`: Adds installed LangChain package versions to tracing metadata after Pydantic initialization.

   The method records the installed `langchain-core` version and, when available, the installed `langchain` version under `metadata["lc_versions"]`.

   Partner model packages should add their versions through uniquely named post-model validators that call `_add_version`, rather than overriding this lifecycle method.

   * **Syntax:**
     ```python
     model_post_init(
         self,
         _context: Any, / # Pydantic validation context
     ) -> None
     ```

2. `_add_version`: Adds or replaces one package version in `metadata["lc_versions"]`.

   Existing package-version mappings are preserved. When `metadata["lc_versions"]` exists but is not a mapping, a warning is emitted and the invalid value is replaced.

   This protected method is intended for subclass and integration validators that need to append their own package version.

   * **Syntax:**
     ```python
     _add_version(
         self,
         pkg: str, # Package name
         version: str # Installed package version
     ) -> None
     ```

3. `set_verbose`: Validates the `verbose` field before model construction.

   A value of `None` is replaced with the current global verbosity setting. Other Boolean values are returned unchanged.

   * **Syntax:**
     ```python
     @field_validator(
         "verbose",
         mode="before"
     )
     set_verbose(
         cls,
         verbose: bool | None # Explicit or inherited verbosity value
     ) -> bool
     ```

4. `generate_prompt`: Generates model results synchronously for a list of prompt values.

   Implementations should use batched provider calls when available. The returned `LLMResult` may contain multiple candidate generations for each prompt and provider-specific output.

   Concrete subclasses must implement this abstract method.

   * **Syntax:**
     ```python
     @abstractmethod
     generate_prompt(
         self,
         prompts: list[
             PromptValue
         ], # Prompt values to process
         stop: list[str] | None = None, # Stop substrings
         callbacks: Callbacks = None, # Callbacks used during generation
         **kwargs: Any # Provider-specific generation parameters
     ) -> LLMResult
     ```

5. `agenerate_prompt`: Asynchronously generates model results for a list of prompt values.

   Implementations should use batched provider calls when available. The returned `LLMResult` may contain multiple candidate generations for each prompt and provider-specific output.

   Concrete subclasses must implement this abstract method.

   * **Syntax:**
     ```python
     @abstractmethod
     async agenerate_prompt(
         self,
         prompts: list[
             PromptValue
         ], # Prompt values to process
         stop: list[str] | None = None, # Stop substrings
         callbacks: Callbacks = None, # Callbacks used during generation
         **kwargs: Any # Provider-specific generation parameters
     ) -> LLMResult
     ```

6. `with_structured_output`: Returns a Runnable configured to produce values matching a schema.

   The schema may be a dictionary or a Python type. The base implementation raises `NotImplementedError`; subclasses that can steer provider output into structured data should override it.

   * **Syntax:**
     ```python
     with_structured_output(
         self,
         schema: dict[
             str,
             Any
         ] | type, # Required output schema
         **kwargs: Any # Model-specific structured-output parameters
     ) -> Runnable[
         LanguageModelInput,
         dict[
             str,
             Any
         ] | BaseModel
     ]
     ```

7. `_get_ls_params`: Returns standard LangSmith parameters for the model invocation.

   The base implementation returns an empty `LangSmithParams` dictionary. Concrete model integrations may override this protected method to include provider, model, temperature, token, stop, and integration metadata.

   * **Syntax:**
     ```python
     _get_ls_params(
         self,
         stop: list[str] | None = None, # Stop strings used for generation
         **kwargs: Any # Invocation parameters
     ) -> LangSmithParams
     ```

8. `_get_ls_params_with_defaults`: Returns LangSmith parameters while allowing subclasses to apply additional defaults.

   The base implementation delegates directly to `_get_ls_params`.

   * **Syntax:**
     ```python
     _get_ls_params_with_defaults(
         self,
         stop: list[str] | None = None, # Stop strings used for generation
         **kwargs: Any # Invocation parameters
     ) -> LangSmithParams
     ```

9. `get_token_ids`: Returns the ordered token IDs for a text value.

   When `custom_get_token_ids` is configured, that encoder is used. Otherwise, the text is encoded using the cached fallback GPT-2 tokenizer.

   The fallback path emits a warning the first time it is used because GPT-2 token counts may be inaccurate for other model families.

   * **Syntax:**
     ```python
     get_token_ids(
         self,
         text: str # Text to tokenize
     ) -> list[int]
     ```

10. `get_num_tokens`: Returns the number of tokens in a text value.

    The base implementation counts the IDs returned by `get_token_ids`. Model integrations should override this method when a model-specific tokenizer is available.

    * **Syntax:**
      ```python
      get_num_tokens(
          self,
          text: str # Text whose tokens are counted
      ) -> int
      ```

11. `get_num_tokens_from_messages`: Returns the combined token count for a list of messages.

    Each message is converted to a buffer string and counted through `get_num_tokens`. The base implementation may include role prefixes added by `get_buffer_string`.

    Tool schemas are not counted. When `tools` is supplied, a warning is emitted and those values are ignored.

    Model integrations should override this method when they support provider-specific message or tool tokenization.

    * **Syntax:**
      ```python
      get_num_tokens_from_messages(
          self,
          messages: list[
              BaseMessage
          ], # Messages whose tokens are counted
          tools: Sequence[Any] | None = None # Tool schemas ignored by the base implementation
      ) -> int
      ```

## Internal Components Omitted

The following private implementation details are not documented as public API entries:

- `_get_token_ids_default_method`
- `_get_verbosity`
- `_get_langchain_version`
- `_GPT2_TOKENIZER_WARNED`
- `_HAS_TRANSFORMERS`